In [1]:
import requests
import json
from datetime import datetime, timezone
from pyspark.sql import Row

In [2]:
env='prod'
print(f"Running on environment: {env}")

Running on environment: prod


In [3]:
def configurations():
    day = datetime.now(timezone.utc).strftime("%m/%d/%Y")
    url_V1 = 'https://statsapi.mlb.com/api/v1/'
    url = f'{url_V1}schedule?sportId=1&date={day}'
    bronze_schema = f'mlb_{env}_bronze'
    table_schedule = f"{bronze_schema}.game_schedule"
    table_failed_game_schedule = f"{bronze_schema}.failed_game_schedule"
    return url, table_schedule, table_failed_game_schedule

url, table_schedule, table_failed_game_schedule = configurations()
url, table_schedule, table_failed_game_schedule

('https://statsapi.mlb.com/api/v1/schedule?sportId=1&date=03/28/2026',
 'mlb_prod_bronze.game_schedule',
 'mlb_prod_bronze.failed_game_schedule')

In [6]:
def get_schedule(url):
    schedule = requests.get(url)
    try:
        schedule.raise_for_status()
        schedule = schedule.json().get('dates', [{}])[0].get('games', [])
        return schedule
    except requests.exceptions.HTTPError as err:
        raise err

schedule = get_schedule(url)
schedule

[{'gamePk': 823082,
  'gameGuid': 'df557aa3-3a78-46ae-ae87-a1bce641c579',
  'link': '/api/v1.1/game/823082/feed/live',
  'gameType': 'R',
  'season': '2026',
  'gameDate': '2026-03-28T18:15:00Z',
  'officialDate': '2026-03-28',
  'status': {'abstractGameState': 'Preview',
   'codedGameState': 'S',
   'detailedState': 'Scheduled',
   'statusCode': 'S',
   'startTimeTBD': False,
   'abstractGameCode': 'P'},
  'teams': {'away': {'team': {'id': 139,
     'name': 'Tampa Bay Rays',
     'link': '/api/v1/teams/139'},
    'leagueRecord': {'wins': 0, 'losses': 1, 'pct': '.000'},
    'splitSquad': False,
    'seriesNumber': 1},
   'home': {'team': {'id': 138,
     'name': 'St. Louis Cardinals',
     'link': '/api/v1/teams/138'},
    'leagueRecord': {'wins': 1, 'losses': 0, 'pct': '1.000'},
    'splitSquad': False,
    'seriesNumber': 1}},
  'venue': {'id': 2889,
   'name': 'Busch Stadium',
   'link': '/api/v1/venues/2889'},
  'content': {'link': '/api/v1/game/823082/content'},
  'gameNumber': 1,

In [5]:
def process_schedule(schedule):
    games_today = []
    failed_responses = []
    for game in schedule:
        try:
            game_id = str(game['gamePk'])
            home_team = game['teams']['home']['team']['name']
            away_team = game['teams']['away']['team']['name']
            game_scheduled_time = game['gameDate']
            game_scheduled_time = datetime.strptime(game_scheduled_time, "%Y-%m-%dT%H:%M:%SZ")
            status = game['status']['abstractGameCode']
            ingestion_time = datetime.now(timezone.utc)
            games_today.append(
                Row(
                    game_pk=game_id,
                    home_team=home_team,
                    away_team=away_team,
                    game_scheduled_time=game_scheduled_time,
                    status=status,
                    ingestion_timestamp=ingestion_time
                )
            )
        except Exception as e:
            failed_responses.append(
                Row(
                    response=json.dumps(game),
                    ingestion_timestamp=datetime.now(timezone.utc)
                )
            )
    return games_today, failed_responses

games_today, failed_responses = process_schedule(schedule)

In [0]:
def save_to_bronze(games_today, failed_responses, table_schedule, table_failed_game_schedule):
    if games_today:
        df_games = spark.createDataFrame(games_today)
        df_games.write.format("delta").mode("append").saveAsTable(table_schedule)
    if failed_responses:
        df_failed = spark.createDataFrame(failed_responses)
        df_failed.write.format("delta").mode("append").saveAsTable(table_failed_game_schedule)